<a href="https://colab.research.google.com/github/vischia/autodifftutorial/blob/master/03_spiking_networks_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# When timing carries information

(C) Pietro Vischia, 2026  

Prepared with assistance from CODEX with GPT-5.6.

Can two sensor records contain exactly the same event counts, yet require different predictions? We will build a small, controlled experiment, train an SNN with `snnTorch`, and compare it with conventional models.

By the end, you should be able to:

1. Explain membrane decay, thresholding, and reset in code.
2. Locate the surrogate gradient and the temporal state in an SNN.
3. Distinguish the value of temporal information from the value of a particular architecture.
4. Report task performance, observation time, and spike activity without turning operation counts into unsupported energy claims.

**You are supposed to work in pairs**. No GPU, external data files, or account is required once the packages are installed.

**Scope:** an intentionally small two-channel event-order problem, not a detector simulation or a state-of-the-art benchmark. It connects directly to the course's encoding, surrogate-gradient, and calorimeter-readout sections. The core code runs as supplied. 

**Structure of the exercises:** Thre are five exercises, which ask you to 1) predict the outcome, 2) change the code, and 3) interpret the results

### Before class: environment

The `requirements.txt` file for `conda` should have installed the relevant dependencies. If you still miss them, or of you are running on Colab and imports are missing, then in the next cell chang to `INSTALL_PACKAGES=True` if imports are missing. Installation may need internet and a kernel restart. After that, the core tutorial runs offline.

For a reproducible local environment, I recommend the `conda`way.


In [ ]:
INSTALL_PACKAGES = False  # Run this BEFORE class if needed.
if INSTALL_PACKAGES:
    import importlib.util, subprocess, sys
    if importlib.util.find_spec("torch") is None:
        command = [sys.executable, "-m", "pip", "install", "torch==2.8.0"]
        if sys.platform != "darwin":
            command += ["--index-url", "https://download.pytorch.org/whl/cpu"]
        subprocess.check_call(command)
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "snntorch==1.0.0", "numpy==2.3.5", "matplotlib==3.10.8"])
    print("Packages installed. Restart the kernel if any packages were already imported.")

In [1]:
import copy
import html
import platform
import time
import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import snntorch as snn
from snntorch import surrogate
from IPython.display import display, HTML

print("Python", platform.python_version(), "| PyTorch", torch.__version__,
      "| snnTorch", snn.__version__, "| NumPy", np.__version__,
      "| Matplotlib", matplotlib.__version__)
if snn.__version__ != "1.0.0":
    warnings.warn("This practical was tested with snnTorch 1.0.0. Check the reset self-test below.")

# A small CPU problem. One thread also reduces timing variability here.
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
T = 40
DT_MS = 1.0  # Illustrative model time, not a calorimeter sampling interval.
HIDDEN = 24
EPOCHS = 18
BATCH_SIZE = 64
NOTEBOOK_START = time.perf_counter()

def show_table(headers, rows):
    def cell(x):
        return html.escape(str(x))
    body = "".join("<tr>" + "".join(f"<td>{cell(v)}</td>" for v in row) + "</tr>" for row in rows)
    head = "".join(f"<th>{cell(h)}</th>" for h in headers)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

Python 3.10.13 | PyTorch 2.5.1 | snnTorch 0.9.4 | NumPy 1.26.4 | Matplotlib 3.7.5


/var/folders/_z/jj9k3dln24d4m13tc_cfv5dh0000gn/T/ipykernel_10257/3634225917.py:19: UserWarning: This practical was tested with snnTorch 1.0.0. Check the reset self-test below.
  warnings.warn("This practical was tested with snnTorch 1.0.0. Check the reset self-test below.")


### 1 · One neuron is a dynamical system [5–20 min]

With voltage measured relative to rest and input scaling absorbed into the current increment, a discrete LIF model is

$$u_t=\beta u_{t-1}+I_t-\theta s_{t-1},\qquad s_t=\mathbb{1}[u_t>\theta],\qquad \beta=e^{-\Delta t/\tau_m}.$$

We use **subtractive reset on the next update**, explicitly setting `reset_delay=True`. The returned voltage can therefore sit above threshold on a spiking step. We omit an explicit refractory period. This is a discrete teaching model, not an exact event-time solver.

**Exercise 1 — predict, edit, explain.** Before running, predict which of `beta=0.5` and `beta=0.95` retains input history longer. Write the update in `lif_step`, check it against the reference, then vary `BETA_SINGLE`. Finally explain why changing the time step while keeping beta fixed changes the physical time constant.

The optional function starts as `None` so the notebook remains runnable. If you leave it unimplemented, the cell explicitly uses the reference. The instructor copy shows a worked implementation.

In [ ]:
# Exercise 1: optionally replace None with your own function:
# def lif_step(u, current, beta, threshold=1.0):
#     ...
#     return new_u, spike
lif_step = None
BETA_SINGLE = 0.85  # Try 0.50 and 0.95 after making a prediction.

def reference_lif_step(u, current, beta, threshold=1.0):
    previous_spike = (u > threshold).to(u.dtype)
    new_u = beta * u + current - threshold * previous_spike
    return new_u, (new_u > threshold).to(u.dtype)

step = reference_lif_step if lif_step is None else lif_step
print("Using", "the supplied reference" if lif_step is None else "your implementation")

currents = torch.zeros(35)
currents[3:15] = 0.36
currents[[21, 24, 27]] = 0.75
u = torch.zeros(1, 1)
manual_u, manual_s = [], []
for current in currents:
    u, s = step(u, current, BETA_SINGLE)
    manual_u.append(u.item()); manual_s.append(s.item())

neuron = snn.Leaky(beta=BETA_SINGLE, threshold=1.0,
                   reset_mechanism="subtract", reset_delay=True)
mem = torch.zeros(1, 1)
lib_u, lib_s = [], []
with torch.no_grad():
    for current in currents:
        s, mem = neuron(current.reshape(1, 1), mem)
        lib_u.append(mem.item()); lib_s.append(s.item())
np.testing.assert_allclose(manual_u, lib_u, atol=1e-6)
np.testing.assert_array_equal(manual_s, lib_s)
print("Hand-written update agrees with snnTorch on this test input.")
print(f"At Δt={DT_MS:g} ms, beta={BETA_SINGLE:g} gives τ={-DT_MS/np.log(BETA_SINGLE):.2f} ms")

fig, ax = plt.subplots(3, 1, figsize=(9, 5.3), sharex=True, constrained_layout=True)
times = np.arange(len(currents)) * DT_MS
ax[0].step(times, currents.numpy(), where="post", color="#4779ad")
ax[0].set_ylabel("Input increment")
ax[1].plot(times, manual_u, "o-", ms=3, color="#4779ad")
ax[1].axhline(1, ls="--", color="#be6740", label="threshold")
ax[1].set_ylabel("Stored voltage"); ax[1].legend(loc="upper right")
ax[2].vlines(times[np.array(manual_s) > 0], 0, 1, color="#be6740")
ax[2].set(xlabel="Model time [ms]", ylabel="Spike", ylim=(-0.05, 1.2))
plt.show()

**Your answer:**

_Write your prediction, result, and explanation here._

### 2 · Same counts, different timing [20–30 min]

Each sample is a binary tensor of shape **[time, channel]** with 40 bins and two sensor channels, A and B.

- Class 0: a burst in A precedes a burst in B.
- Class 1: the order is reversed.
- Onset, separation, burst size, and background events vary.

We generate **pairs** with opposite labels and identical per-channel counts. Each pair shares its nuisance realization. We generate the train, validation, and test sets independently, at the pair level. The hidden generator parameters are never model inputs.

**Exercise 2.** Explain why any deterministic classifier that receives only the two total counts has exactly 50% accuracy on a complete paired split, regardless of its architecture or training. What would go wrong if related records from one physical shower were scattered across train and test?

In [ ]:
def make_dataset(n_pairs, seed):
    rng = np.random.default_rng(seed)
    x = np.zeros((2 * n_pairs, T, 2), dtype=np.float32)
    y = np.tile([0, 1], n_pairs).astype(np.int64)
    for p in range(n_pairs):
        onset = int(rng.integers(4, 10))
        gap = int(rng.integers(10, 18))
        n_signal = int(rng.integers(2, 5))
        offsets = np.sort(rng.choice(4, n_signal, replace=False))
        early, late = onset + offsets, onset + offsets + gap
        # Avoid collisions that could change binary event counts between labels.
        quiet = np.setdiff1d(np.arange(T), np.r_[early, late])
        background = np.zeros((T, 2), dtype=np.float32)
        for channel in range(2):
            ts = rng.choice(quiet, int(rng.integers(0, 3)), replace=False)
            background[ts, channel] = 1
        x[2*p:2*p+2] = background
        x[2*p, early, 0] = 1; x[2*p, late, 1] = 1
        x[2*p+1, early, 1] = 1; x[2*p+1, late, 0] = 1
    return torch.from_numpy(x), torch.from_numpy(y)

X_train, y_train = make_dataset(256, 11)
X_val, y_val = make_dataset(64, 22)
X_test, y_test = make_dataset(128, 33)
for x, y in [(X_train, y_train), (X_val, y_val), (X_test, y_test)]:
    assert torch.equal(x[0::2].sum(1), x[1::2].sum(1))
    assert torch.all(y[0::2] == 0) and torch.all(y[1::2] == 1)
print("Train / validation / test examples:", len(X_train), len(X_val), len(X_test))
print("Every opposite-label pair has identical per-channel counts.")

fig, axes = plt.subplots(2, 3, figsize=(11, 4.5), sharex=True, sharey=True,
                          constrained_layout=True)
for p in range(3):
    for label in [0, 1]:
        sample = X_train[2*p+label].numpy()
        axes[label, p].eventplot([np.flatnonzero(sample[:, c])*DT_MS for c in [0, 1]],
                                 lineoffsets=[0, 1], linelengths=0.55,
                                 colors=["#4779ad", "#be6740"])
        axes[label, p].set_title(f"Pair {p+1}, class {label}; counts {sample.sum(0).astype(int)}")
        axes[label, p].set_yticks([0, 1], ["A", "B"])
        axes[label, p].set_xlim(0, T*DT_MS)
        if label == 1: axes[label, p].set_xlabel("Model time [ms]")
plt.show()

**Your answer:**

_Write your prediction, result, and explanation here._

### 3 · Binary forward pass, surrogate backward pass [30–40 min]

The step function has zero ordinary derivative away from threshold and no ordinary derivative at threshold. We keep its **binary forward output** while specifying a useful backward derivative. For snnTorch's fast-sigmoid surrogate,

$$\widetilde{\frac{\partial s}{\partial u}}=\frac{1}{(1+k|u-\theta|)^2}.$$

**Exercise 3.** Change `SURROGATE_SLOPE` and predict what happens to the forward spikes and to the range of voltages with an appreciable gradient. Is this the exact gradient of the original discrete system?

This is a deliberate approximation. In the `Leaky` configuration used below, snnTorch also detaches the reset signal in the backward pass. Temporal state still carries gradients; this is not the same as detaching the membrane at every step. See the source links in the references.

In [ ]:
SURROGATE_SLOPE = 10.0  # Try 3.0 and 30.0.
overdrive = torch.linspace(-1, 1, 501, requires_grad=True)
spike_fn = surrogate.fast_sigmoid(slope=SURROGATE_SLOPE)
spikes = spike_fn(overdrive)
spikes.sum().backward()
assert set(spikes.detach().unique().tolist()) <= {0.0, 1.0}
expected_gradient = 1 / (1 + SURROGATE_SLOPE * overdrive.detach().abs())**2
torch.testing.assert_close(overdrive.grad, expected_gradient)

fig, ax = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)
ax[0].plot(overdrive.detach(), spikes.detach(), color="#4779ad")
ax[0].set(xlabel="Voltage above threshold", ylabel="Forward spike", title="Still a hard threshold")
ax[1].plot(overdrive.detach(), overdrive.grad, color="#be6740")
ax[1].set(xlabel="Voltage above threshold", ylabel="Backward derivative", title="Surrogate derivative")
plt.show()

**Your answer:**

_Write your prediction, result, and explanation here._

### 4 · Build and train a small SNN [40–60 min]

The model has **2 inputs → 24 LIF neurons → 2 class logits**. The arrows denote signal flow. There is no recurrent weight matrix, but each LIF neuron has temporal state. A filtered spike readout supplies additional fading memory:

$$r_t=\alpha r_{t-1}+s_t,\qquad z=W_{\rm out}r_{T-1}+b_{\rm out},\quad \alpha=0.95.$$

The two real-valued logits feed cross-entropy; the output layer is deliberately nonspiking. Readout choice is part of the model. Both membrane and readout state reset to zero at the start of each independent example.

**Before running:** point to the code lines implementing the leak, surrogate, state reset, and readout. Explain why there is no `mem.detach()` inside the time loop. Predict whether setting `beta=0` removes *all* temporal memory.

Training uses training data only. Validation selects the checkpoint; we reserve the test split for the final comparisons. This is a small classroom demonstration, not a hyperparameter search or a publication-quality benchmark.

In [ ]:
class TemporalSNN(nn.Module):
    def __init__(self, beta=0.85, readout_beta=0.95, slope=10.0):
        super().__init__()
        self.input = nn.Linear(2, HIDDEN)
        self.lif = snn.Leaky(beta=beta, threshold=1.0,
                             spike_grad=surrogate.fast_sigmoid(slope=slope),
                             reset_mechanism="subtract", reset_delay=True)
        self.output = nn.Linear(HIDDEN, 2)
        self.readout_beta = readout_beta
        # Start with enough input drive for useful spiking; no learned result is encoded.
        with torch.no_grad():
            self.input.weight.uniform_(-0.2, 1.0)
            self.input.bias.zero_()

    def forward(self, x):  # x: [batch, time, channels]
        mem = x.new_zeros((x.shape[0], HIDDEN))
        readout = torch.zeros_like(mem)
        spike_history = []
        for t in range(x.shape[1]):
            current = self.input(x[:, t])
            spikes, mem = self.lif(current, mem)
            readout = self.readout_beta * readout + spikes
            spike_history.append(spikes)
        logits = self.output(readout)
        return logits, torch.stack(spike_history, dim=1)

@torch.no_grad()
def accuracy(model, x, y):
    model.eval()
    return (model(x)[0].argmax(1) == y).float().mean().item()

def fit(factory, train, validation, *, seed=SEED, epochs=EPOCHS, activity_weight=0.0):
    torch.manual_seed(seed)
    model = factory().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    generator = torch.Generator().manual_seed(123)
    x, y = train
    vx, vy = validation
    history = {"loss": [], "val_accuracy": []}
    best_accuracy, best_epoch, best_state = -1.0, None, None
    start = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        loss_sum = 0.0
        for indices in torch.randperm(len(x), generator=generator).split(BATCH_SIZE):
            optimizer.zero_grad(set_to_none=True)
            logits, spikes = model(x[indices])
            loss = nn.functional.cross_entropy(logits, y[indices])
            if spikes is not None:
                loss = loss + activity_weight * spikes.mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += float(loss.detach()) * len(indices)
        val_accuracy = accuracy(model, vx, vy)
        history["loss"].append(loss_sum / len(x))
        history["val_accuracy"].append(val_accuracy)
        if val_accuracy > best_accuracy:
            best_accuracy, best_epoch = val_accuracy, epoch + 1
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    elapsed = time.perf_counter() - start
    print(f"{type(model).__name__}: {elapsed:.2f} s training; "
          f"best validation {best_accuracy:.1%} at epoch {best_epoch}")
    return model, history

snn_model, snn_history = fit(TemporalSNN, (X_train, y_train), (X_val, y_val))
# Confirm that the first-layer weights receive a gradient through the surrogate.
snn_model.zero_grad(set_to_none=True)
logits, _ = snn_model(X_train[:16])
nn.functional.cross_entropy(logits, y_train[:16]).backward()
assert snn_model.input.weight.grad is not None
print("Input-weight gradient norm:", float(snn_model.input.weight.grad.norm()))
snn_model.zero_grad(set_to_none=True)

fig, ax = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)
epoch_axis = np.arange(1, EPOCHS+1)
ax[0].plot(epoch_axis, snn_history["loss"], color="#4779ad")
ax[0].set(xlabel="Epoch", ylabel="Mean training objective")
ax[1].plot(epoch_axis, snn_history["val_accuracy"], color="#be6740")
ax[1].axhline(0.5, ls="--", color="gray")
ax[1].set(xlabel="Epoch", ylabel="Validation accuracy", ylim=(0.4, 1.02))
plt.show()

### 5 · Controls: what did we actually establish? [60–75 min]

We need two different conventional comparisons:

- A **count-only linear classifier** deliberately loses time. It tests the information in that representation.
- A **small GRU** sees the same event tensor as the SNN. It tests whether a conventional temporal model can also solve the problem.

Both use the same train/validation/test splits, training epochs, optimizer, and batch size. They are **not parameter-matched or exhaustively tuned**. Report their sizes and resist making general architecture rankings from this exercise.

**Exercise 4.** Predict all three test accuracies before running. If the GRU matches or beats the SNN, does that invalidate temporal encoding? What additional evidence would establish a hardware advantage?

In [ ]:
class CountClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 2)
    def forward(self, x):
        counts = x.sum(dim=1)  # Exercise: identify precisely what this discards.
        return self.linear(counts), None

class TemporalGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(2, 12, batch_first=True)
        self.output = nn.Linear(12, 2)
    def forward(self, x):
        _, hidden = self.gru(x)
        return self.output(hidden[-1]), None

count_model, count_history = fit(CountClassifier, (X_train, y_train), (X_val, y_val))
gru_model, gru_history = fit(TemporalGRU, (X_train, y_train), (X_val, y_val))
models = {"Count-only": count_model, "SNN": snn_model, "GRU": gru_model}

# Each evaluation pair is a cluster, not two independent samples.
def paired_bootstrap_interval(correct, seed=2026):
    pair_accuracy = correct.detach().cpu().numpy().reshape(-1, 2).mean(axis=1)
    rng = np.random.default_rng(seed)
    samples = rng.choice(pair_accuracy, size=(2000, len(pair_accuracy)), replace=True).mean(axis=1)
    return np.quantile(samples, [0.025, 0.975])

results = []
with torch.no_grad():
    for name, model in models.items():
        pred = model(X_test)[0].argmax(1)
        correct = (pred == y_test).float()
        low, high = paired_bootstrap_interval(correct)
        n_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
        results.append([name, n_parameters, f"{correct.mean().item():.1%}",
                        f"{low:.1%}–{high:.1%}"])
assert accuracy(count_model, X_test, y_test) == 0.5
show_table(["Model", "Trainable parameters", "Test accuracy", "95% paired bootstrap interval"], results)

#### A count-preserving retiming control

Rebuild the event trains from their per-channel counts with random event times. Use **the same random reconstruction for both members of a pair**. The opposite labels now have exactly identical inputs. This deliberately constructed information-erasure control makes 50% accuracy unavoidable for any deterministic classifier on the complete paired test set.

This is stronger and more specific than simply shuffling time once at test time. An ordinary post-training shuffle can also introduce distribution shift. Here, the loss of label information follows from the construction, not from an observed accuracy drop alone.

In [ ]:
def erase_order_in_pairs(x, seed=444):
    rng = np.random.default_rng(seed)
    retimed = torch.zeros_like(x)
    assert len(x) % 2 == 0
    assert torch.equal(x[0::2].sum(1), x[1::2].sum(1))
    for p in range(len(x) // 2):
        for c in range(2):
            n = int(x[2*p, :, c].sum())
            times = rng.choice(x.shape[1], n, replace=False)
            retimed[2*p:2*p+2, times, c] = 1
    assert torch.equal(retimed.sum(1), x.sum(1))
    assert torch.equal(retimed[0::2], retimed[1::2])
    return retimed

X_test_retimed = erase_order_in_pairs(X_test)
comparison = []
for name, model in models.items():
    original = accuracy(model, X_test, y_test)
    erased = accuracy(model, X_test_retimed, y_test)
    assert erased == 0.5
    comparison.append([name, f"{original:.1%}", f"{erased:.1%}"])
show_table(["Model", "Original timing", "Timing erased, counts retained"], comparison)

**Your answer:**

_Write your prediction, result, and explanation here._

### 6 · Activity, observation time, and implementation cost [75–85 min]

Inspect a hidden-layer raster. We will report **spikes per example** and the **fraction of active neuron-time bins**. The latter is not a rate in hertz; converting to hertz requires the model time step.

For a *hypothetical* sparse implementation of the two fully connected projections, an event-operation proxy is

$$N_{\rm syn}\approx N_{\rm input\ spikes}\times 24 + N_{\rm hidden\ spikes}\times 2.$$

This excludes state decay, bias processing, routing, memory, encoding, and readout overhead. Our actual PyTorch implementation executes dense operations in a Python time loop. This proxy therefore describes neither its instruction count nor its energy.

**Predict:** can two networks emit the same number of spikes but have different communication costs? What measurements would turn an activity estimate into joules per decision?

In [ ]:
with torch.no_grad():
    test_logits, hidden_spikes = snn_model(X_test)
input_spikes_per_example = X_test.sum((1, 2)).mean().item()
hidden_spikes_per_example = hidden_spikes.sum((1, 2)).mean().item()
active_fraction = hidden_spikes.mean().item()
event_proxy = input_spikes_per_example * HIDDEN + hidden_spikes_per_example * 2
show_table(["Quantity", "Value"], [
    ["Input events / example", f"{input_spikes_per_example:.2f}"],
    ["Hidden spikes / example", f"{hidden_spikes_per_example:.2f}"],
    ["Active hidden neuron-time bins", f"{active_fraction:.2%}"],
    ["Mean rate per hidden neuron at the illustrative time scale", f"{active_fraction/(DT_MS/1000):.1f} Hz"],
    ["Hypothetical projection event-operations / example", f"{event_proxy:.1f}"],
])

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)
for j in [0, 1]:
    spikes_j = hidden_spikes[j].numpy()
    ax[j].eventplot([np.flatnonzero(spikes_j[:, i])*DT_MS for i in range(HIDDEN)],
                    colors="#4779ad", linelengths=0.7)
    ax[j].set(xlabel="Model time [ms]", ylabel="Hidden neuron", xlim=(0, T*DT_MS),
              title=f"Test pair 1, class {j}, prediction {int(test_logits[j].argmax())}")
plt.show()

In [ ]:
# Prefix sensitivity of models trained for the FULL window, not separately optimized early classifiers.
cutoffs = [8, 16, 24, 32, 40]
prefix_scores = {name: [accuracy(model, X_test[:, :n], y_test) for n in cutoffs]
                 for name, model in models.items()}
fig, ax = plt.subplots(figsize=(7.5, 3.5), constrained_layout=True)
for name, scores in prefix_scores.items():
    ax.plot(np.array(cutoffs)*DT_MS, scores, "o-", label=name)
ax.axhline(0.5, ls="--", color="gray")
ax.set(xlabel="Available observation window [model ms]", ylabel="Test accuracy",
       ylim=(0.0, 1.02), title="Prefix sensitivity, without retraining")
ax.legend()
plt.show()

# CPU execution timing only, batch size one, excluding data collection.
@torch.no_grad()
def cpu_latency_ms(model, sample, repeats=30):
    model.eval()
    for _ in range(5):
        model(sample)
    elapsed = []
    for _ in range(repeats):
        start = time.perf_counter()
        model(sample)
        elapsed.append(1000 * (time.perf_counter() - start))
    return float(np.median(elapsed))

show_table(["Model", "Median CPU execution / sample [ms]"],
           [[name, f"{cpu_latency_ms(model, X_test[:1]):.3f}"] for name, model in models.items()])
print("Observation window:", T*DT_MS, "model ms. CPU timings exclude observation time.")

Record one sentence for each distinction:

1. The count-only model may exceed chance on a **prefix**, even though full-window counts cannot predict the label. Why?
2. An SNN can use few spikes yet run slower than a GRU in this notebook. Why?
3. Lowering an activity count is not equivalent to lowering measured energy. Why?

The prefix experiment changes the input distribution of a model trained for full records. It is a sensitivity study, not an optimal early-exit algorithm. The timing cell is a small CPU demonstration, not a device benchmark. A streaming implementation may overlap processing with observation; report those boundaries explicitly.

### 7 · Transfer to the calorimeter case study [85–90 min]

The course's readout example encodes photon counts with threshold channels. The toy inputs here are already binary events, but the same modeling decisions reappear.

**Exercise 5 — exit ticket.** In five minutes, answer:

1. For energy regression, what would you change in the output layer, target scaling, and loss?
2. Name one comparison that tests whether timing improves reconstruction and one that tests whether the SNN architecture helps.
3. At what physical level would you split the data to avoid leakage?
4. Which observation, processing, and measurement boundaries would you report before claiming a latency or energy advantage?

**Hand in:** your five short exercise answers, the three-model comparison table, and one plot with an interpretation. A good conclusion may favor a conventional model.

**End of the 90-minute core.**

**Your answer:**

_Write your prediction, result, and explanation here._

### Optional extensions (either to do at home, or today if you have time)

Choose one of the following extensions, and use the validation data to make choices before a fresh final test.

- **Activity regularization:** compare the same initial seed with $\mathcal L=\mathcal L_{\rm CE}+\lambda\,\mathrm{mean}(s)$. Report accuracy and spikes together. Activity is still an energy proxy.
- **Temporal constants:** change beta and the readout decay separately. A beta-only ablation leaves other memory pathways.
- **Training variability:** repeat several initialization seeds and report the spread. The paired bootstrap above does not capture this variability.
- **Sensor representation:** for a new toy photon signal, implement threshold channels as in the lecture. Keep the physical model and its assumptions separate from the SNN.

The first two experiments below are ready to run. They use validation data only, to avoid turning repeated classroom exploration into test-set selection.

In [ ]:
RUN_EXTENSIONS = False
if RUN_EXTENSIONS:
    extension_rows = []
    for activity_weight in [0.0, 0.1, 1.0]:
        candidate, _ = fit(TemporalSNN, (X_train, y_train), (X_val, y_val),
                           activity_weight=activity_weight)
        with torch.no_grad():
            logits, spikes = candidate(X_val)
        extension_rows.append([activity_weight,
                               f"{(logits.argmax(1)==y_val).float().mean().item():.1%}",
                               f"{spikes.sum((1,2)).mean().item():.2f}"])
    show_table(["Activity penalty λ", "Validation accuracy", "Hidden spikes / example"], extension_rows)

    time_constant_rows = []
    for beta, readout_beta in [(0.0, 0.95), (0.85, 0.0), (0.95, 0.95)]:
        candidate, _ = fit(lambda: TemporalSNN(beta=beta, readout_beta=readout_beta),
                           (X_train, y_train), (X_val, y_val))
        time_constant_rows.append([beta, readout_beta, f"{accuracy(candidate, X_val, y_val):.1%}"])
    show_table(["Membrane beta", "Readout alpha", "Validation accuracy"], time_constant_rows)
else:
    print("Extensions skipped. The 90-minute core is complete.")

### References and next steps

This notebook uses original teaching code and synthetic data. Its biological and computational context comes from the course and these sources:

1. Eshraghian et al., **Training Spiking Neural Networks Using Lessons From Deep Learning**, Proceedings of the IEEE (2023). [Paper](https://arxiv.org/abs/2109.12894) · [snnTorch tutorial 5](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_5.html).
2. Neftci, Mostafa & Zenke, **Surrogate Gradient Learning in Spiking Neural Networks**, IEEE Signal Processing Magazine (2019). [Paper](https://arxiv.org/abs/1901.09948).
3. snnTorch 1.0.0: [Leaky documentation](https://snntorch.readthedocs.io/en/latest/snn.neurons_leaky.html), [Leaky source](https://snntorch.readthedocs.io/en/latest/_modules/snntorch/_neurons/leaky.html), [reset source](https://snntorch.readthedocs.io/en/latest/_modules/snntorch/_neurons/neurons.html), and [surrogate API](https://snntorch.readthedocs.io/en/latest/snntorch.surrogate.html). The explicit reset self-test takes priority over ambiguous timing shorthand in prose documentation.
4. Lupi et al., **Neuromorphic Readout for Hadron Calorimeters**, Particles 8(2), 52 (2025). [Published paper](https://doi.org/10.3390/particles8020052)
5. Yik et al., **The NeuroBench framework for benchmarking neuromorphic computing algorithms and systems**, Nature Communications (2025). [Paper](https://doi.org/10.1038/s41467-025-56739-4)

If you want to study more of this, then replace the synthetic task with [Spiking Heidelberg Digits](https://zenkelab.org/resources/spiking-heidelberg-datasets-shd/), or a carefully documented detector dataset, and retain the same controls. For physical deployment, follow the course's [Rockpool/Xylo training tutorial](https://rockpool.ai/devices/torch-training-spiking-for-xylo.html) and verify that model/reset conventions match before transferring weights.